# Text-to-Cypher: original recorded Colab run
Saved outputs preserved, not re-executed. Failed attempts and installation duplicates removed; API key replaced with environment-based authentication. Colab Drive paths require your saved artifacts. Install the recorded environment; API judging incurs charges.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import Dataset, load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch, json, random

ds = load_dataset("tomasonjo/text2cypher-gpt4o-clean", split="train")

test_databases = {'bluesky', 'stackoverflow2'}
test_data = [ex for ex in ds if ex['database'] in test_databases]
train_val_data = [ex for ex in ds if ex['database'] not in test_databases]

random.seed(42)
random.shuffle(test_data)
random.shuffle(train_val_data)

test = test_data[:100]
val = train_val_data[:100]
train = train_val_data[100:900]

def format_example(row):
    return {
        "instruction": (
            "You are a Cypher query expert. Given a Neo4j graph schema and a "
            "natural language question, write the correct Cypher query.\n\n"
            f"Schema:\n{row['schema']}\n\n"
            f"Question: {row['question']}"
        ),
        "output": row["cypher"]
    }

train_formatted = [format_example(ex) for ex in train]
val_formatted = [format_example(ex) for ex in val]
test_formatted = [format_example(ex) for ex in test]

with open('/content/drive/MyDrive/train_formatted.json', 'w') as f:
    json.dump(train_formatted, f)
with open('/content/drive/MyDrive/val_formatted.json', 'w') as f:
    json.dump(val_formatted, f)
with open('/content/drive/MyDrive/test_set.json', 'w') as f:
    json.dump(test_formatted, f)

print(f"Train: {len(train_formatted)}, Val: {len(val_formatted)}, Test: {len(test_formatted)}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def format_for_training(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

train_ds = Dataset.from_list(train_formatted).map(format_for_training)
val_ds = Dataset.from_list(val_formatted).map(format_for_training)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    dataset_text_field = "text",
    max_seq_length = 2048,
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,
        save_strategy = "no",
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("Starting training...")
trainer.train()

model.save_pretrained("/content/drive/MyDrive/cypher-lora-final")
tokenizer.save_pretrained("/content/drive/MyDrive/cypher-lora-final")
print("Done! Model saved to Google Drive.")

Mounted at /content/drive
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Train: 800, Val: 100, Test: 100
==((====))==  Unsloth 2026.8.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.8.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 3 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.588088,0.378088
100,0.035104,0.033213
150,0.019879,0.029878
200,0.024536,0.028916
250,0.021975,0.028960
300,0.021734,0.028692


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/cypher-lora-final/tokenizer_config.json.


Done! Model saved to Google Drive.


In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch, json

# Load fine-tuned model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/cypher-lora-final",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# Load one test example
with open('/content/drive/MyDrive/test_set.json') as f:
    test_set = json.load(f)

example = test_set[0]

# Generate output
messages = [{"role": "user", "content": example["instruction"]}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.0, do_sample=False)

pred = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print("QUESTION:", example["instruction"].split("Question:")[-1].strip())
print("\nGOLD:", example["output"])
print("\nPREDICTED:", pred)
print("\nMATCH:", pred.strip() == example["output"].strip())

==((====))==  Unsloth 2026.8.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


QUESTION: Show the first 5 questions that have answers from users who have also asked questions.

GOLD: MATCH (q:Question)<-[:ANSWERED]-(a:Answer)<-[:PROVIDED]-(u:User)-[:ASKED]->(q2:Question)
RETURN q
LIMIT 5

PREDICTED: MATCH (u:User)-[:ASKED]->(q:Question)<-[:PROVIDED]-(a:Answer)
RETURN q.title AS question_title, u.display_name AS user_name
ORDER BY q.creation_date ASC
LIMIT 5

MATCH: False


In [ ]:
def get_prediction(model, tokenizer, instruction):
    messages = [{"role": "user", "content": instruction}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        output = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            temperature=0.0,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip()

# Eval fine-tuned model
ft_results = []
for i, ex in enumerate(test_set):
    pred = get_prediction(model, tokenizer, ex["instruction"])
    match = pred.strip() == ex["output"].strip()
    ft_results.append({"instruction": ex["instruction"], "gold": ex["output"], "pred": pred, "match": match})
    if i % 10 == 0:
        print(f"{i}/100 done...")

ft_score = sum(r["match"] for r in ft_results) / len(ft_results)
print(f"\nFine-tuned exact match: {ft_score:.1%}")

with open('/content/drive/MyDrive/ft_results.json', 'w') as f:
    json.dump(ft_results, f)

0/100 done...


10/100 done...


20/100 done...


30/100 done...


40/100 done...


...

60/100 done...


70/100 done...


80/100 done...


90/100 done...



Fine-tuned exact match: 10.0%


In [ ]:
# Load BASE model (no adapter)
del model
torch.cuda.empty_cache()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)
base_tokenizer = get_chat_template(base_tokenizer, chat_template="qwen-2.5")

base_results = []
for i, ex in enumerate(test_set):
    pred = get_prediction(base_model, base_tokenizer, ex["instruction"])
    match = pred.strip() == ex["output"].strip()
    base_results.append({"instruction": ex["instruction"], "gold": ex["output"], "pred": pred, "match": match})
    if i % 10 == 0:
        print(f"{i}/100 done...")

base_score = sum(r["match"] for r in base_results) / len(base_results)
print(f"\nBase model exact match: {base_score:.1%}")

with open('/content/drive/MyDrive/base_results.json', 'w') as f:
    json.dump(base_results, f)

==((====))==  Unsloth 2026.8.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


0/100 done...


10/100 done...


20/100 done...


30/100 done...


...

60/100 done...


70/100 done...


80/100 done...


90/100 done...



Base model exact match: 0.0%


In [ ]:
from openai import OpenAI

client = OpenAI()  # Configure OPENAI_API_KEY securely

with open('/content/drive/MyDrive/base_results.json') as f:
    base_results = json.load(f)
with open('/content/drive/MyDrive/ft_results.json') as f:
    ft_results = json.load(f)

JUDGE_PROMPT = """You are evaluating Cypher query generation.

Question: {question}
Gold query: {gold}
Model A output: {a}
Model B output: {b}

Which output is more correct and complete? Reply with just "A", "B", or "tie". Then on a new line, one sentence explaining why."""

judge_results = []
a_wins, b_wins, ties = 0, 0, 0

for i, (base, ft) in enumerate(zip(base_results, ft_results)):
    question = base["instruction"].split("Question:")[-1].strip()

    # Randomly assign base/ft to A/B to avoid bias
    import random
    if random.random() > 0.5:
        a, b, a_is_base = base["pred"], ft["pred"], True
    else:
        a, b, a_is_base = ft["pred"], base["pred"], False

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            question=question, gold=base["gold"], a=a, b=b
        )}],
        max_tokens=100,
    )

    verdict = response.choices[0].message.content.strip()
    winner = verdict.split("\n")[0].strip()

    if winner == "A":
        ft_won = not a_is_base
    elif winner == "B":
        ft_won = a_is_base
    else:
        ft_won = None

    if ft_won is True: a_wins += 1
    elif ft_won is False: b_wins += 1
    else: ties += 1

    judge_results.append({"question": question, "verdict": verdict, "ft_won": ft_won})

    if i % 10 == 0:
        print(f"{i}/100 done... FT winning: {a_wins}, Base winning: {b_wins}, Ties: {ties}")

print(f"\nFinal — Fine-tuned wins: {a_wins}% | Base wins: {b_wins}% | Ties: {ties}%")

with open('/content/drive/MyDrive/judge_results.json', 'w') as f:
    json.dump(judge_results, f)

0/100 done... FT winning: 0, Base winning: 1, Ties: 0
10/100 done... FT winning: 6, Base winning: 4, Ties: 1
20/100 done... FT winning: 13, Base winning: 7, Ties: 1
30/100 done... FT winning: 21, Base winning: 8, Ties: 2
40/100 done... FT winning: 25, Base winning: 13, Ties: 3
50/100 done... FT winning: 34, Base winning: 14, Ties: 3
60/100 done... FT winning: 40, Base winning: 18, Ties: 3
70/100 done... FT winning: 47, Base winning: 21, Ties: 3
80/100 done... FT winning: 56, Base winning: 22, Ties: 3
90/100 done... FT winning: 62, Base winning: 25, Ties: 4

Final — Fine-tuned wins: 68% | Base wins: 28% | Ties: 4%


In [ ]:
from datasets import load_dataset
import random

# Load 200 MMLU questions
mmlu = load_dataset("cais/mmlu", "all", split="test").shuffle(seed=42).select(range(200))

def eval_mmlu(model, tokenizer, data):
    correct = 0
    choices = ["A", "B", "C", "D"]

    for i, ex in enumerate(data):
        prompt = f"Question: {ex['question']}\nA) {ex['choices'][0]}\nB) {ex['choices'][1]}\nC) {ex['choices'][2]}\nD) {ex['choices'][3]}\nAnswer with just the letter."

        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            output = model.generate(input_ids=inputs, max_new_tokens=5, temperature=0.0, do_sample=False)

        pred = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip().upper()
        gold = choices[ex['answer']]

        if pred.startswith(gold):
            correct += 1

        if i % 50 == 0:
            print(f"{i}/200 done...")

    return correct / len(data)

# Load fine-tuned model
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/cypher-lora-final",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

ft_mmlu = eval_mmlu(model, tokenizer, mmlu)
print(f"Fine-tuned MMLU: {ft_mmlu:.1%}")

# Load base model
del model
torch.cuda.empty_cache()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)
base_tokenizer = get_chat_template(base_tokenizer, chat_template="qwen-2.5")

base_mmlu = eval_mmlu(base_model, base_tokenizer, mmlu)
print(f"Base MMLU: {base_mmlu:.1%}")

print(f"\nDifference: {(ft_mmlu - base_mmlu):.1%}")

==((====))==  Unsloth 2026.8.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


0/200 done...


50/200 done...


100/200 done...


150/200 done...


...

0/200 done...


50/200 done...


100/200 done...


150/200 done...


Base MMLU: 57.5%

Difference: 1.5%


In [ ]:
import json, shutil

# Verify all files exist in Drive
import os
files = [
    '/content/drive/MyDrive/cypher-lora-final',
    '/content/drive/MyDrive/test_set.json',
    '/content/drive/MyDrive/ft_results.json',
    '/content/drive/MyDrive/base_results.json',
    '/content/drive/MyDrive/judge_results.json',
]

for f in files:
    exists = os.path.exists(f)
    print(f"{'✓' if exists else '✗'} {f}")

✓ /content/drive/MyDrive/cypher-lora-final
✓ /content/drive/MyDrive/test_set.json
✓ /content/drive/MyDrive/ft_results.json
✓ /content/drive/MyDrive/base_results.json
✓ /content/drive/MyDrive/judge_results.json
